# 📧 Email Spam Detection (Teaching Version - Colab Ready)
End-to-End AI Pipeline

## 🔑 Step 1: Setup Kaggle API

In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()  # Upload kaggle.json

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)

In [ ]:
!cp kaggle.json /root/.kaggle/

In [ ]:
!chmod 600 /root/.kaggle/kaggle.json

## 📥 Step 2: Download Dataset

In [ ]:
!kaggle datasets download -d uciml/sms-spam-collection-dataset

In [ ]:
!unzip -o sms-spam-collection-dataset.zip

## 📊 Step 3: Load Dataset

In [ ]:
import pandas as pd
df = pd.read_csv('spam.csv', encoding='latin-1')

In [ ]:
df.head()

## 🧹 Step 4: Data Cleaning

In [ ]:
df = df[['v1','v2']]

In [ ]:
df.columns = ['label','message']

In [ ]:
df['label'] = df['label'].map({'ham':0,'spam':1})

In [ ]:
df.info()

## 📈 Step 5: Exploratory Data Analysis

In [ ]:
df['label'].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.countplot(x='label', data=df)
plt.show()

In [ ]:
df['length'] = df['message'].apply(len)

In [ ]:
df[['length']].describe()

In [ ]:
sns.histplot(df['length'], bins=50)
plt.show()

## ☁️ Step 6: WordCloud Visualization

In [ ]:
from wordcloud import WordCloud

In [ ]:
spam_words = ' '.join(df[df['label']==1]['message'])

In [ ]:
wc = WordCloud(width=800,height=400).generate(spam_words)

In [ ]:
plt.imshow(wc)
plt.axis('off')
plt.show()

## 🔤 Step 7: Text Preprocessing

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:

def clean_text(text):
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower().split()
    text = [w for w in text if w not in stop_words]
    return ' '.join(text)


In [ ]:
df['clean'] = df['message'].apply(clean_text)

In [ ]:
df[['message','clean']].head()

## 🔢 Step 8: TF-IDF Feature Engineering

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(max_features=3000)

In [ ]:
X = tfidf.fit_transform(df['clean']).toarray()

In [ ]:
y = df['label']

## 🔀 Step 9: Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

## 🤖 Step 10: Logistic Regression Model

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression(max_iter=200)

In [ ]:
model.fit(X_train,y_train)

## 📊 Step 11: Model Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [ ]:
pred = model.predict(X_test)

In [ ]:
accuracy_score(y_test,pred)

In [ ]:
confusion_matrix(y_test,pred)

In [ ]:
print(classification_report(y_test,pred))

## 📉 Step 12: ROC Curve

In [ ]:
from sklearn.metrics import roc_curve, auc

In [ ]:
probs = model.predict_proba(X_test)[:,1]

In [ ]:
fpr,tpr,_ = roc_curve(y_test,probs)

In [ ]:
plt.plot(fpr,tpr)
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.show()

## 🧠 Step 13: Deep Learning Model (ANN)

In [ ]:
from tensorflow.keras.models import Sequential

In [ ]:
from tensorflow.keras.layers import Dense

In [ ]:
ann = Sequential()

In [ ]:
ann.add(Dense(128,activation='relu',input_dim=X_train.shape[1]))

In [ ]:
ann.add(Dense(64,activation='relu'))

In [ ]:
ann.add(Dense(1,activation='sigmoid'))

In [ ]:
ann.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])

In [ ]:
ann.fit(X_train,y_train,epochs=3,validation_data=(X_test,y_test))

In [ ]:
ann.evaluate(X_test,y_test)

## 🔮 Step 14: Test Custom Input

In [ ]:

def predict_spam(text):
    cleaned = clean_text(text)
    vec = tfidf.transform([cleaned]).toarray()
    return model.predict(vec)[0]

predict_spam("Congratulations! You won a free lottery ticket")
